In [1]:
#!pip install sympy

In [2]:
from pyomo.environ import *
import numpy as np
from math import pi
from Sympy_OPF_LALM_class import SympyACOPFModel
from Sympy_OPF_LALM_class import PrintQHDACOPFResults
from Sympy_OPF_LALM_class import solve_with_gurobi_from_sympy
from Sympy_OPF_LALM_class import initialize_qhd_acopf_log
from Sympy_OPF_LALM_class import append_qhd_acopf_results
from qhdopt import QHD
import json

In [3]:
def load_matpower_json(json_file):
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    Sbase = float(data["Sbase"])

    # 把 "k1", "k2", ... 转成 int key: 1, 2, ...
    buses = {int(k.replace("k", "")): v for k, v in data["buses"].items()}
    lines = {int(k.replace("k", "")): v for k, v in data["lines"].items()}
    gens  = {int(k.replace("k", "")): v for k, v in data["gens"].items()}

    return Sbase, buses, lines, gens

In [4]:
# 初始化（用默认 3-bus 数据）
#model = SympyACOPFModel()   # 打开 proximal 选项，后面会用到
n = 3  # 选择测试系统的规模，2 或 3 或者更多

if n == 2:
    # 2bus model
    Sbase = 10.0
    buses = {
        1: [1, 0, 1.00, 0.0, 0.0, 0.0, 0.0, 0.0],
        2: [2, 1, 1.01, 0.0, 0.0, 0.0, 0.3, 0.1],
    }
    lines = {
        1: [1, 2, 0.0452, 0.1852, 0.0204, 1.0, 30.0 / Sbase],
    }
    gens = {
        1: [1, 0.0 / Sbase, 20.0 / Sbase, -20.0 / Sbase, 100.0 / Sbase, 0.00375, 2.0, 0.0],
    }
    model = SympyACOPFModel(Sbase = Sbase, buses=buses, lines=lines, gens=gens)

elif n == 3:
    # 3bus model
    model = SympyACOPFModel()
else:
    # n bus model
    Sbase, buses, lines, gens = load_matpower_json(f"case{n}_custom_pretty.json")
    model = SympyACOPFModel(Sbase = Sbase, buses=buses, lines=lines, gens=gens) # 打开 proximal 选项，后面会用到

print("Model initialized with", n, "buses", model.n_lines, "lines and", model.n_gens, "generators.")


Model initialized with 3 buses 3 lines and 2 generators.


In [5]:
# =========================
#   Linear ALM + QHD Loop
# =========================

import numpy as np

# 1) 构造 h(x)
h_func = model.build_h_func()
model.reset_lambdas(0.0)

# 2) 初始点
xk = model.build_initial_x0()
#xk = np.load("opf_result_ordered.npy", allow_pickle=True)

rho = 5.0
alpha = 5e-3   # 对偶步长尽量小一点
max_outer = 2
tol = 1e-4

# ========= 新增：初始化日志文件 =========
log_file = initialize_qhd_acopf_log(model, folder="logs")
print("Log file:", log_file)

print("\n===== Start Linear ALM Loop =====\n")

for k in range(max_outer):

    print(f"\n--- Outer Iteration {k} ---")

    # ======================================
    # (1) 在线性化点 xk 构造二次 L^(k)(x)
    # ======================================
    Lag, variable_list, Var_bound_list = \
        model.build_linear_ALM_Lagrangian_syms(
            x_center=xk,
            rho=rho,
            ref_bus_id=None,
            mu_prox=10.0
        )
    
    bad_bounds = []
    for i, (var, bnd) in enumerate(zip(variable_list, Var_bound_list)):
        lb, ub = float(bnd[0]), float(bnd[1])
        if ub < lb:
            bad_bounds.append((i, str(var), lb, ub))

    if bad_bounds:
        print("\n=== Invalid bounds found ===")
        for item in bad_bounds:
            print(item)
        raise ValueError("Var_bound_list contains invalid bounds (ub < lb).")

    option = 1  # 1: QHD, 2: Gurobi
    if option == 1:
        # ======================================
        # (2A) QHD 求解子问题
        # ======================================
        qhd_model = QHD.SymPy(Lag, variable_list, Var_bound_list)


        qhd_solver = "simbi" # openjij / simbi
        if qhd_solver == "simbi":
            qhd_model.simbi_setup(
                resolution=7,
                agents=2048,
                max_steps=3000,
                embedding_scheme="unary",
                post_processing_method='TNC',
                verbose=True
            )
        else:
            qhd_model.openjij_setup(
                resolution=6,
                shots=1024,
                sampler_name="SQASampler",
                seed=42,
                debug=False,
                sampler_init_kwargs={},
                sample_kwargs={
                    "beta": 5.0,
                    "gamma": 1.0,
                    "trotter": 8,
                    "num_sweeps": 3000,
                    "reinitialize_state": True,
                },
            )
        response = qhd_model.optimize(refine=True, verbose=0)

        x_new = np.asarray(response.refined_minimizer)

    else:
        # ======================================
        # (2B) 用 Gurobi 求解子问题
        # ======================================
        x_new = solve_with_gurobi_from_sympy(
        L_sym=Lag,
        variable_list=variable_list,
        Var_bound_list=Var_bound_list,
        verbose=False   # 如果想看 Gurobi 日志就改成 True
    )
    PrintQHDACOPFResults(model, x_new)

    # ======================================
    # (3) 计算真实约束
    # ======================================
    h_val = h_func(x_new)
    norm_h = np.linalg.norm(h_val)

    print("||h(x)|| =", norm_h)

    # ======================================
    # (4) 对偶更新
    # ======================================
    lambda_new, h_x = model.update_lambda(
        x_new,
        alpha=rho,
        h_func=h_func
    )
    # ======================================
    # (5) 自适应 rho
    # ======================================
    h_old = h_func(xk)
    print(f"[rho-check] ||h_old||={np.linalg.norm(h_old):.3e}, ||h_new||={np.linalg.norm(h_val):.3e}, rho={rho:.3g}")
    rho_max = 1024
    if np.linalg.norm(h_x) > 0.5 * np.linalg.norm(h_old) and rho < rho_max:
        rho *= 2
        print("Increasing rho to", rho)

    # ======================================
    # (6) 可行性检查
    # ======================================
    _, check_flag = model.check_constraints(xk)
    print("Constraint check:", check_flag)
    if check_flag:
        print("\nConverged!")
        break

    # ======================================
    # (7) 记录日志（每轮追加）
    # ======================================
    subs_dict = {var: val for var, val in zip(model.variable_list, x_new)}
    #objective_value = float(model.objective.evalf(subs=subs_dict))

    append_qhd_acopf_results(
        model=model,
        solution=x_new,
        log_file=log_file,
        iteration=k,
        rho=rho,
        alpha=alpha,
        h_x=h_val,
        lambda_vec=lambda_new,
        objective_value=0,
        feasibility=check_flag,
    )

    # 如果你还想同时在屏幕上打印表格，可以再保留这一句
    # PrintQHDACOPFResults(model, x_new, iteration=k, log_file=log_file, print_to_screen=True)


    # ======================================
    # (8) 收敛判据
    # ======================================
    step_norm = np.linalg.norm(x_new - xk)
    if norm_h < tol and step_norm < 1e-5:
        print("\nConverged!")
        break

    # ======================================
    # (9) 更新 primal
    # ======================================
    xk = x_new.copy()

print("\n===== End Loop =====\n")
print("Final log file:", log_file)

Log file: logs\Buses-3-18-36-02-03-31-2026.txt

===== Start Linear ALM Loop =====


--- Outer Iteration 0 ---


Bifurcated agents:   0%|          | 0/2048 [00:01<?, ?it/s]


backend time consumption: 1.62326979637146
Minimizer: [0.71428571 0.85714286 0.71428571 0.71428571 0.85714286 0.85714286
 0.71428571 0.85714286 0.85714286 0.85714286 0.85714286 0.85714286
 0.85714286 0.71428571 0.71428571 0.71428571 0.85714286 0.85714286
 0.71428571 0.85714286 0.71428571 0.71428571 0.71428571 0.85714286
 0.85714286 0.71428571 0.85714286 0.71428571 0.85714286 0.71428571
 0.71428571]
Update time: 2026-03-31 18:37:11
------------------------------------------------------------------------------------------------------------------------
Bus Results
BusID	VR		VI		Vmag		Pg		Qg		Pl		Ql		Qshunt
1	1.009409	0.041287	1.005032	0.000000	0.051331	0.000000	0.000000	0.000000
2	1.008455	0.041703	1.009622	0.000000	0.026697	0.000000	0.000000	0.000000
3	0.999816	0.032067	1.000727	0.000000	0.000000	0.300000	0.100000	0.000000

Branch Results
LineID	From	To	Pik		Pki		Qik		Qki		LossP		LossQ
1	1	2	0.003216	0.001859	0.022992	-0.055522	0.005075	-0.032531
2	1	3	0.003862	-0.000155	0.011268	-0.0529

Bifurcated agents:   0%|          | 0/2048 [00:01<?, ?it/s]


backend time consumption: 1.6668968200683594
Minimizer: [0.85714286 0.71428571 0.71428571 0.71428571 0.71428571 0.71428571
 0.71428571 0.85714286 0.71428571 0.85714286 0.85714286 0.85714286
 0.71428571 0.85714286 0.85714286 0.71428571 0.85714286 0.71428571
 0.71428571 0.85714286 0.71428571 0.71428571 0.85714286 0.71428571
 0.85714286 0.71428571 0.71428571 0.71428571 0.71428571 0.71428571
 0.71428571]
Update time: 2026-03-31 18:38:14
------------------------------------------------------------------------------------------------------------------------
Bus Results
BusID	VR		VI		Vmag		Pg		Qg		Pl		Ql		Qshunt
1	0.983695	-0.035082	1.022326	0.000000	0.043327	0.000000	0.000000	0.000000
2	0.981868	-0.037353	1.018463	0.000000	-0.013936	0.000000	0.000000	0.000000
3	0.973190	-0.044843	1.025693	0.000000	0.000000	0.300000	0.100000	0.000000

Branch Results
LineID	From	To	Pik		Pki		Qik		Qki		LossP		LossQ
1	1	2	0.007388	0.017860	0.036711	-0.005684	0.025248	0.031027
2	1	3	0.095671	-0.046547	0.045699	-0

In [18]:
model.arc_to_line

[1, 1, 2, 2, 3, 3]

In [7]:
x_new

array([ 0.        ,  0.        ,  0.04332707, -0.01393637,  0.98369545,
        0.98186793,  0.9731898 , -0.03508151, -0.0373532 , -0.04484267,
        1.04515049,  1.03726688,  1.05204524,  0.00738825,  0.01785994,
        0.0956705 , -0.04654749,  0.04798038, -0.04503525,  0.03671084,
       -0.00568424,  0.04569913, -0.05300753,  0.00899703, -0.02288265,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ])

In [8]:
Lag, variable_list, Var_bound_list = \
        model.build_linear_ALM_Lagrangian_syms(
            x_center=np.ones_like(xk),
            rho=rho,
            ref_bus_id=None,
            mu_prox=1500.0
        )

In [9]:
Lag

0.001875*P_G0**2 + 1.18068362626373*P_G0 + 0.00875*P_G1**2 + 1.37258756738426*P_G1 - 0.141076430866968*P_ij0 + 0.346228535086962*P_ij1 + 0.00568575920525012*P_ij2 + 0.447668105314362*P_ij3 - 0.390105129839027*P_ij4 + 0.315811932485676*P_ij5 + 0.152143186319155*Q_G0 + 0.139842632847636*Q_G1 + 0.157616340664994*Q_ij0 - 0.112099779805431*Q_ij1 + 0.00505640248875776*Q_ij2 - 0.0679435299966107*Q_ij3 - 0.179374228679984*Q_ij4 + 0.181256499832171*Q_ij5 - 0.00970615162685218*S_tot_sq0 - 0.0171874517066339*S_tot_sq1 - 0.0569156513613842*S_tot_sq2 - 0.0388762636827398*S_tot_sq3 - 0.0133859295019806*S_tot_sq4 - 0.0140782962075888*S_tot_sq5 + 5.0*V_I0**2 + 10.9016739220643*V_I0 - 21.4038548032732*V_I1 + 8.12261947903981*V_I2 - 16.4248941130528*V_R0 + 18.1693897207257*V_R1 - 4.15508450242867*V_R2 + 0.328710253319524*V_sq0 + 0.362118391424833*V_sq1 + 0.518655138994329*V_sq2 + 750.0*(P_G0 - 1.0)**2 + 750.0*(P_G1 - 1.0)**2 + 750.0*(P_ij0 - 1.0)**2 + 750.0*(P_ij1 - 1.0)**2 + 750.0*(P_ij2 - 1.0)**2 + 75

In [10]:
model.linear_jacobian

Matrix([
[1, 0, 0, 0, 15.6467268408034*V_I1 + 5.09602092120209*V_I2 - 12.9367669346993*V_R0 + 5.22464617988566*V_R1 + 1.24373728746401*V_R2,                                                                         -15.6467268408034*V_I0 + 5.22464617988566*V_R0,                                                                         -5.09602092120209*V_I0 + 1.24373728746401*V_R0, -12.9367669346993*V_I0 + 5.22464617988566*V_I1 + 1.24373728746401*V_I2 - 15.6467268408034*V_R1 - 5.09602092120209*V_R2,                                                                         5.22464617988566*V_I0 + 15.6467268408034*V_R0,                                                                         1.24373728746401*V_I0 + 5.09602092120209*V_R0, 0, 0, 0,        0,        0,        0,        0,        0,        0,        0,        0,        0,        0,        0,        0, 0, 0, 0, 0, 0, 0],
[0, 1, 0, 0,                                                                        -15.6467268408034*V_I1 + 5.22

In [11]:
model.G_mat

array([[ 6.46838347, -5.22464618, -1.24373729],
       [-5.22464618,  6.9301765 , -1.70553032],
       [-1.24373729, -1.70553032,  2.9492676 ]])

In [12]:
model.arc_collection

[(0, 1), (1, 0), (0, 2), (2, 0), (1, 2), (2, 1)]

In [13]:
response.refined_minimizer


array([ 0.        ,  0.        ,  0.04332707, -0.01393637,  0.98369545,
        0.98186793,  0.9731898 , -0.03508151, -0.0373532 , -0.04484267,
        1.04515049,  1.03726688,  1.05204524,  0.00738825,  0.01785994,
        0.0956705 , -0.04654749,  0.04798038, -0.04503525,  0.03671084,
       -0.00568424,  0.04569913, -0.05300753,  0.00899703, -0.02288265,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ])

In [14]:
qhd_model.func

5.001875*P_G0**2 + 1.72159197273331*P_G0 + 5.00875*P_G1**2 + 1.41788831137777*P_G1 + 0.0273081495709372*P_ij0 - 0.00188743735055679*P_ij1 - 0.270305003725081*P_ij2 + 0.287739991820974*P_ij3 - 0.387018304987945*P_ij4 + 0.244112917208622*P_ij5 + 0.0890963858969155*Q_G0 + 0.176385827715959*Q_G1 + 0.0291305965873494*Q_ij0 - 0.193458860832974*Q_ij1 - 0.144663042703266*Q_ij2 - 0.0694906240946209*Q_ij3 - 0.102414397486679*Q_ij4 + 0.122227100670466*Q_ij5 + 5.0*S_tot_sq0**2 - 0.00269479003179659*S_tot_sq0 + 5.0*S_tot_sq1**2 - 0.0154310107150131*S_tot_sq1 + 5.0*S_tot_sq2**2 - 0.00070937505009252*S_tot_sq2 + 5.0*S_tot_sq3**2 - 0.0139939274144245*S_tot_sq3 + 5.0*S_tot_sq4**2 - 0.0014706124175535*S_tot_sq4 + 5.0*S_tot_sq5**2 - 0.00131935075411643*S_tot_sq5 + 2.5*V_I0**2 - 0.695392195944952*V_I0 + 1.11038895046*V_I1 - 0.206870103207571*V_I2 - 2.78963978640482*V_R0 + 2.61413898904761*V_R1 + 0.274183250948973*V_R2 - 0.052604946510153*V_sq0 + 0.00308342122760763*V_sq1 + 0.00397513579750708*V_sq2 + 5.0*

In [15]:
print(model.P_D, model.Q_D, model.P_G, model.Q_G)
print(model.variable_list)
print(model.Var_bound_list)

[0.  0.  0.3] [0.  0.  0.1] (P_G0, P_G1) (Q_G0, Q_G1)
[P_G0, P_G1, Q_G0, Q_G1, V_R0, V_R1, V_R2, V_I0, V_I1, V_I2, V_sq0, V_sq1, V_sq2, P_ij0, P_ij1, P_ij2, P_ij3, P_ij4, P_ij5, Q_ij0, Q_ij1, Q_ij2, Q_ij3, Q_ij4, Q_ij5, S_tot_sq0, S_tot_sq1, S_tot_sq2, S_tot_sq3, S_tot_sq4, S_tot_sq5]
[[0.0, 2.0], [0.0, 2.0], [-2.0, 10.0], [-2.0, 10.0], [-1.1, 1.1], [-1.1, 1.1], [-1.1, 1.1], [-1.1, 1.1], [-1.1, 1.1], [-1.1, 1.1], [0.81, 1.2100000000000002], [0.81, 1.2100000000000002], [0.81, 1.2100000000000002], [-3.0, 3.0], [-3.0, 3.0], [-3.0, 3.0], [-3.0, 3.0], [-3.0, 3.0], [-3.0, 3.0], [-3.0, 3.0], [-3.0, 3.0], [-3.0, 3.0], [-3.0, 3.0], [-3.0, 3.0], [-3.0, 3.0], [0.0, 9.0], [0.0, 9.0], [0.0, 9.0], [0.0, 9.0], [0.0, 9.0], [0.0, 9.0]]
